In [1]:
# ============================================================
# PopQA EM Evaluation — OpenAI Version (Clean & Stable)
# ============================================================

from __future__ import annotations
import os
import time
from contextlib import nullcontext

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm
from datasets import load_dataset

from fuseqa import *

import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

# ------------------------------------------------------------
# Load environment
# ------------------------------------------------------------
load_dotenv()

client = OpenAI()
MODEL_NAME = "gpt-5-nano" #"gpt-5.4"   # or "gpt-5-nano"

print(f"Using model: {MODEL_NAME}")




Using model: gpt-5-nano


In [2]:
# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
ds = load_dataset("MinaGabriel/entityquestions-retrieval20-with-sre")["train"]#.select(range(10))

In [3]:
def prompt_fn(question: str, context: str, use_context: bool):
    if use_context:
        return (
            f"Context: {context}\n\n"
            f"Q: {question}\n"
            "A (one word from context):"
        )
    else:
        return (
            f"Q: {question}\n"
            "A (one word only):"
        )

In [5]:

# ------------------------------------------------------------
# OpenAI LLM Call
# ------------------------------------------------------------
def ask_llm_openai(question, context, use_context, print_prompt=False):
    prompt = prompt_fn(question, context, use_context)

    if print_prompt:
        print("=== PROMPT ===")
        print(prompt)

    resp = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        max_output_tokens=32,
        reasoning={"effort": "minimal"}
    )

    pred = (resp.output_text or "").strip().rstrip(".")

    if print_prompt:
        print("=== PREDICTION ===")
        print(pred)

    return pred


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
TOP_K = 3
CLIP_CHARS = 1025
SRE_SCORE_TH = 0.90

RUN_TYPE = "NO-CONTEXT"  # "FUSEQA", "FUSEQA-SRE", "NO-CONTEXT"
USE_CONTEXT = RUN_TYPE in ("FUSEQA", "FUSEQA-SRE")

RESULTS_DIR = "results"
WRITE_OUTPUTS = True
PRINT_PROMPTS = False

os.makedirs(RESULTS_DIR, exist_ok=True)

file_name = hf_model_to_filename(MODEL_NAME + "-" + RUN_TYPE)
outfile = os.path.join(RESULTS_DIR, file_name + ".jsonl")

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
counts  = {g: 0 for g in ("ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT")}
em_hits = {g: 0 for g in ("ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT")}

def update_metrics(tier, em):
    counts["ALL"] += 1
    em_hits["ALL"] += em
    if tier in counts:
        counts[tier] += 1
        em_hits[tier] += em

def current_scores():
    return {
        "ALL_EM":     safe_div(em_hits["ALL"],        counts["ALL"]),
        "Long_Tail":  safe_div(em_hits["LONG-TAIL"],  counts["LONG-TAIL"]),
        "Infrequent": safe_div(em_hits["INFREQUENT"], counts["INFREQUENT"]),
        "Frequent":   safe_div(em_hits["FREQUENT"],   counts["FREQUENT"]),
    }

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
start_time = time.time()
tqdm._instances.clear()

with (open(outfile, "w", encoding="utf-8", buffering=1) if WRITE_OUTPUTS else nullcontext()) as writer:
    with tqdm(
        total=len(ds),
        desc="Generating + Evaluating (OpenAI)",
        dynamic_ncols=True,
        leave=False,
    ) as pbar:

        for i in range(len(ds)):
            ex = ds[i]

            q = ex["question"]
            s_pop = int(ex.get("s_pop_avg", 0))
            tier = tier_from_spop(s_pop)

            gold = parse_list(ex.get("possible_answers"))
            gold_norm_set = {norm(g) for g in gold if norm(g)}

            context = get_context_for_run(
                ex, RUN_TYPE, USE_CONTEXT,
                top_k=TOP_K,
                sre_score_th=SRE_SCORE_TH,
                clip_chars=CLIP_CHARS
            )

            pred = ask_llm_openai(
                question=q,
                context=context,
                use_context=USE_CONTEXT,
                print_prompt=PRINT_PROMPTS,
            )

            pred_norm = norm(pred)
            em = int(pred_norm in gold_norm_set) if gold_norm_set else 0

            update_metrics(tier, em)

            if WRITE_OUTPUTS:
                write_record(writer, {
                    "i": i,
                    "s_pop": s_pop,
                    "tier": tier,
                    "question": q,
                    "gold": gold,
                    "pred": pred,
                    "em": em,
                })

            pbar.update(1)

            if i % 10 == 0:
                pbar.set_postfix({k: f"{v:.4f}" for k, v in current_scores().items()})

total_time = time.time() - start_time

generate_report(
    counts,
    em_hits,
    file_name,
    model_name=MODEL_NAME,
    run_type=RUN_TYPE,
    total_time=total_time,
)

Generating + Evaluating (OpenAI):   0%|          | 0/22075 [00:00<?, ?it/s]

INFO:openai._base_client:Retrying request to /responses in 0.459779 seconds
INFO:openai._base_client:Retrying request to /responses in 0.384514 seconds


Saved report: results/gpt-5-nano-NO-CONTEXT_20260307-1910.report.txt | Time=20329.94s | ALL=0.1745 | LONG-TAIL=0.1657 | INFREQUENT=0.1785 | FREQUENT=0.1911


'results/gpt-5-nano-NO-CONTEXT_20260307-1910.report.txt'